# Building Image Generation Applications (Azure OpenAI)

There's more to LLMs than text generation. You can also generate images from text descriptions. Images as a modality are useful across MedTech, architecture, tourism, game development, marketing, and more. In this lesson we work with today's **GPT Image** models on Microsoft Foundry.

## Learning goals

By the end of this lesson you'll be able to:

- Explain what image generation is and where it's useful.
- Understand the `gpt-image` model family and how it differs from the legacy DALL·E models.
- Build an image generation application and edit images.

## What is image generation?

Image generation models create images from a text prompt. Modern models such as `gpt-image` learn the relationship between text and images during training, then iteratively turn random noise into an image that matches your prompt.

Two well-known families of image models are:

- **`gpt-image` (OpenAI)** — the current generation used in this lesson. It supports text-to-image generation and image editing (inpainting with a mask).
- **Midjourney** — a popular third-party model with its own service and Discord-based workflow.

> The older OpenAI image models — **DALL·E 2** and **DALL·E 3** — are legacy. DALL·E 3 is no longer available for new deployments, and features like `create_variation` only existed in DALL·E 2. Use the `gpt-image` models for new applications.

On Microsoft Foundry, **`gpt-image-2`** is the latest and most capable image model and is the recommended default. `gpt-image-1.5` and `gpt-image-1-mini` are also generally available.

> **Important:** `gpt-image` models return the generated image as **base64** (`b64_json`), not as a URL. Your code decodes the base64 string to bytes and saves it — there's no image URL to download.


## Vytvoření vaší první aplikace pro generování obrázků

Co je tedy potřeba k vytvoření aplikace pro generování obrázků? Potřebujete následující knihovny:

- **python-dotenv**, tuto knihovnu se velmi doporučuje používat k uchování vašich tajemství v souboru *.env* mimo kód.
- **openai**, tuto knihovnu použijete k interakci s OpenAI API.
- **pillow**, pro práci s obrázky v Pythonu.

Pokud jste tak ještě neučinili, postupujte podle pokynů na stránce [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) k vytvoření zdroje a modelu Microsoft Foundry. Jako model vyberte **gpt-image-2** (nejnovější model Azure OpenAI pro obrázky; DALL·E je zastaralý).

1. Vytvořte soubor *.env* s následujícím obsahem:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    Tyto informace najdete v portálu Microsoft Foundry pro váš zdroj v sekci "Deployments".



1. Shromážděte výše uvedené knihovny do souboru nazvaného *requirements.txt* takto:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Dále vytvořte virtuální prostředí a nainstalujte knihovny:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Pro Windows použijte následující příkazy k vytvoření a aktivaci vašeho virtuálního prostředí:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Přidejte následující kód do souboru nazvaného *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # import dotenv
    dotenv.load_dotenv()

    # nakonfigurujte klienta Azure OpenAI (Microsoft Foundry).
    # Modely obrazů vyžadují aktuální verzi API — zkontrolujte dokumentaci Foundry pro verzi, kterou váš model vyžaduje.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # Vytvořte obrázek pomocí API pro generování obrazů
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Sem zadejte text svého promptu
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # např. "gpt-image-2"
        )
        # Nastavte adresář pro uložený obrázek
        image_dir = os.path.join(os.curdir, 'images')

        # Pokud adresář neexistuje, vytvořte jej
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Inicializujte cestu k obrázku (pozor, typ souboru by měl být png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # Modely gpt-image vracejí obrázek jako base64 (b64_json), ne jako URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Zobrazte obrázek v výchozím prohlížeči obrázků
        image = Image.open(image_path)
        image.show()

    # zachytit výjimky
    except BadRequestError as err:
        print(err)

    ```

Vysvětlíme si tento kód:

- Nejprve importujeme knihovny, které potřebujeme, včetně knihovny OpenAI, knihovny dotenv, modulu base64 a knihovny Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- Dále načteme proměnné prostředí ze souboru *.env*.

    ```python
    # import dotenv
    dotenv.load_dotenv()
    ```

- Poté nakonfigurujeme klienta Azure OpenAI (Microsoft Foundry).

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- Následně generujeme obrázek:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Zadejte zde svůj text dotazu
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    Modely `gpt-image` vracejí obrázek jako **base64** řetězec v `data[0].b64_json`. Dekódujeme jej do bytů a zapíšeme do souboru — neexistuje URL ke stažení.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Nakonec otevřeme obrázek a zobrazíme jej ve standardním prohlížeči obrázků:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Podrobnosti o generování obrázku

Podívejme se na parametry funkce `images.generate`:

- **prompt** je textový požadavek použitý k vygenerování obrázku. Zde je to "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size** je velikost generovaného obrázku (například `1024x1024`, `1536x1024`, `1024x1536` nebo `"auto"`).
- **n** je počet generovaných obrázků. Zde generujeme jeden.
- **model** je název vašeho nasazení modelu pro obrázky (například `gpt-image-2`).

> Modely pro obrázky neberou parametr `temperature` — to je parametr pro generování textu. Pro větší rozmanitost voláte API znovu; pro menší rozmanitost udělejte svůj prompt specifičtější.

## Další možnosti generování obrázků

Viděli jste, jak generovat obrázek několika řádky Pythonu. Modely `gpt-image` také mohou **editovat** existující obrázek. Poskytnutím obrázku, volitelné **masky** (která označuje oblast k úpravě) a promptu můžete změnit část obrázku — například přidat klobouk našemu zajíci.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# úpravy jsou také vráceny jako base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Základní obrázek obsahuje jen králíka; finální obrázek má na králíkovi klobouk.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Prohlášení o omezení odpovědnosti**:
Tento dokument byl přeložen pomocí AI překladatelské služby [Co-op Translator](https://github.com/Azure/co-op-translator). Přestože usilujeme o co největší přesnost, mějte prosím na paměti, že automatizované překlady mohou obsahovat chyby nebo nepřesnosti. Originální dokument v jeho mateřském jazyce by měl být považován za autoritativní zdroj. Pro kritické informace se doporučuje profesionální lidský překlad. Nejsme odpovědní za jakékoli nedorozumění nebo nesprávné interpretace vzniklé použitím tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
